[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BSLJunhyeonJeon/AI_COP/blob/main/session3/notebooks/02_build_html.ipynb)

# session3 · 02 · 강의자료 HTML 빌더 (수업 전에 1회 실행)

- **이 노트북에서 하는 것**: `01_finetune` 을 실행해 `outputs/` 를 채우고, 강의 템플릿에 그림을 **base64 인라인** → **단일 파일** `session3_lecture.html` 생성.
- **입력**: `session3/session3_lecture_template.html`(claude.ai 제공) + 01 실행 결과.
- **출력**: `outputs/session3_lecture.html` (외부 연결 없는 자체완결 — 다운로드해서 사용).

> ⚠️ **파이프라인**입니다. **셀 1 → 2 → 3 → 4 → 5 순서대로** 실행하세요.
> ⚠️ **셀 3은 01 의 학습(무료 T4 기준 10~15분)을 포함**합니다. 빌더 전체가 그만큼 걸립니다. **GPU 런타임**으로 실행하세요.

In [ ]:
# 셀 1 · 환경 감지 + 프로젝트 루트 확보 (session2 와 동일 패턴 — 분기는 이 셀 한 곳)
import os, subprocess

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

SESSION = "session3"
REPO_URL = "https://github.com/BSLJunhyeonJeon/AI_COP"
REPO_DIR = "/content/AI_COP"
SESSION_DIR = REPO_DIR + "/" + SESSION


def acquire_project():
    if os.path.isdir(REPO_DIR):
        print("이미 존재:", REPO_DIR, "(재클론 건너뜀)")
        try:
            r = subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=False)
            if r.returncode != 0:
                print("  (git pull 실패 — 기존 캐시 버전 사용)")
        except Exception as e:
            print("  (git pull 건너뜀:", e, ")")
    else:
        print("레포 클론:", REPO_URL, "->", REPO_DIR)
        try:
            subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
        except Exception as e:
            print("clone 실패(네트워크/권한 확인):", e)
    return SESSION_DIR if os.path.isdir(SESSION_DIR) else None


def find_root_local(marker="requirements.txt"):
    start = os.path.abspath(os.getcwd())
    d = start
    while True:
        if os.path.exists(os.path.join(d, marker)):
            return d
        parent = os.path.dirname(d)
        if parent == d:
            print("[주의] '" + marker + "' 를 못 찾음. 현재 폴더를 루트로 가정:", start)
            return start
        d = parent


PROJECT_ROOT = acquire_project() if IN_COLAB else find_root_local()
if not (PROJECT_ROOT and os.path.isdir(PROJECT_ROOT)):
    raise RuntimeError(
        "세션 루트를 확보하지 못했습니다. "
        "코랩이면 레포 클론 실패이니 네트워크 확인 후 이 셀(셀 1)을 다시 ▶ 실행하세요. "
        "로컬이면 session3/ 안에서 노트북을 열었는지 확인하세요."
    )
os.chdir(PROJECT_ROOT)
print("실행 환경   :", "Colab" if IN_COLAB else "Local")
print("PROJECT_ROOT:", PROJECT_ROOT)

In [ ]:
# 셀 2 · 의존성 설치 + 실행 도구 확인 (nbconvert)
import os, sys, subprocess

if not os.path.exists("requirements.txt"):
    raise RuntimeError("requirements.txt 를 찾지 못했습니다. 셀 1을 먼저 ▶ 실행하세요.")
print("requirements.txt 설치 중...")
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=False)
if r.returncode != 0:
    raise RuntimeError("pip install 실패. 위 로그 확인 후 이 셀(셀 2)을 다시 ▶ 실행하세요.")

try:
    import nbconvert
    print("nbconvert:", nbconvert.__version__)
except ImportError:
    print("nbconvert 미설치 -> 설치")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "nbconvert"], check=False)

import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "없음 (학습이 느리고 결과가 약합니다)")

In [ ]:
# 셀 3 · 01_finetune 실행 (같은 VM) — 학습 포함이라 10~15분 걸립니다.
import os, sys, subprocess

NB = "01_finetune.ipynb"
src = os.path.join("notebooks", NB)
if not os.path.exists(src):
    raise RuntimeError("노트북이 없습니다: " + src)

exec_dir = os.path.abspath(os.path.join("outputs", "_executed"))   # 실행본 사본(비커밋)
os.makedirs(exec_dir, exist_ok=True)

print("=== 실행:", NB, "(학습 포함 — 셀당 최대 1800초) ===")
cmd = [sys.executable, "-m", "jupyter", "nbconvert", "--to", "notebook", "--execute",
       "--ExecutePreprocessor.timeout=1800",
       "--output-dir", exec_dir, "--output", NB.replace(".ipynb", "_executed.ipynb"), src]
try:
    cp = subprocess.run(cmd, check=False)
    status = "성공" if cp.returncode == 0 else "실패(exit %d)" % cp.returncode
except Exception as e:
    status = "예외: %s" % e
print("->", status)

EXPECTED = ["03_curves.png", "03_before_after.png",
            "03_conf_010.png", "03_conf_025.png", "03_conf_050.png", "03_conf_070.png",
            "03_per_class.png"]
present = [f for f in EXPECTED if os.path.exists(os.path.join("outputs", f))]
missing = [f for f in EXPECTED if f not in present]
print("--- outputs/ 그림 점검 ---  있음 %d / %d" % (len(present), len(EXPECTED)))
if missing:
    print("  누락:", missing, "(빌드는 누락 자리표시로 진행됩니다)")

In [ ]:
# 셀 4 · HTML 조립 — 템플릿의 data-embed 자리에 outputs/ 그림을 base64 src 로 주입
# (템플릿의 내용/구조/CSS 는 건드리지 않는다. src 속성만 추가. 누락 파일은 그대로 둠.)
import os, re, base64

TEMPLATE = "session3_lecture_template.html"      # session3/ 루트 기준 (claude.ai 제공)
if not os.path.exists(TEMPLATE):
    raise RuntimeError(
        "강의 템플릿이 없습니다: " + TEMPLATE + "\n"
        "이 파일은 claude.ai 가 작성해 session3/ 에 커밋합니다. "
        "템플릿을 받은 뒤 이 셀을 다시 ▶ 실행하세요."
    )
with open(TEMPLATE, encoding="utf-8") as _f:
    html = _f.read()

embedded, missing = [], []


def inject(m):
    fname = m.group(1)
    path = os.path.join("outputs", fname)
    if os.path.exists(path):
        mime = "image/gif" if fname.lower().endswith(".gif") else "image/png"
        with open(path, "rb") as _img:
            b64 = base64.b64encode(_img.read()).decode("ascii")
        embedded.append(fname)
        return 'src="data:%s;base64,%s" data-embed="%s"' % (mime, b64, fname)
    missing.append(fname)
    return m.group(0)                            # 누락: 원본 그대로(템플릿 자리표시 노출)


out_html = re.sub(r'data-embed="([^"]+)"', inject, html)
os.makedirs("outputs", exist_ok=True)
OUT = os.path.join("outputs", "session3_lecture.html")
with open(OUT, "w", encoding="utf-8") as f:
    f.write(out_html)

print("임베드:", len(embedded), "개", embedded)
print("누락  :", len(missing), "개", missing if missing else "")
print("저장:", OUT)

In [ ]:
# 셀 5 · 검증 / 다운로드 안내
import os

OUT = os.path.join("outputs", "session3_lecture.html")
print("=" * 56)
print(" session3 · 강의 HTML 빌드 검증")
print("=" * 56)
if os.path.exists(OUT):
    print(" - 최종 파일 :", OUT)
    print(" - 크기      : %.2f MB" % (os.path.getsize(OUT) / 1e6))
    print("=" * 56)
    print("이 파일을 다운로드해 브라우저로 열면, 외부 연결 없이 단독으로 보이는 강의 자료입니다.")
    try:
        import google.colab  # noqa: F401
        print("(코랩) 좌측 파일창에서 outputs/session3_lecture.html 우클릭 -> 다운로드,")
        print("       또는 새 셀에: from google.colab import files; files.download('%s')" % OUT)
    except ImportError:
        print("(로컬) 위 경로의 파일을 브라우저로 열면 됩니다.")
else:
    print(" [주의] 최종 HTML이 없습니다. 셀 1->2->3->4 를 순서대로 실행했는지 확인하세요.")